In [2]:
 import os
from dotenv import load_dotenv

load_dotenv(override=True)

token = os.environ.get("OPENAI_API_KEY")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"OPENAI_API_KEY loaded ({len(token)} characters): {masked}")
else:
    print("OPENAI_API_KEY not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'OPENAI_API_KEY', and load_dotenv() ran without error.")

OPENAI_API_KEY loaded (164 characters): sk-p...VmEA


In [3]:
"""
Fix entity-type ambiguities by filling in resolved_type on
entity_type_lookup.csv (from find_entity_type_ambiguities.py) using an LLM,
instead of leaving it at plain majority vote.

Why: majority vote picks the most COMMON label, not necessarily the CORRECT
one -- for near-tie names in particular (majority share < 65%), a 51/49
split is close to a coin flip and frequency alone isn't good evidence. An
LLM given the entity name, the taxonomy definitions, and a few real source
sentences per candidate type can make a domain judgment call instead.

This edits entity_type_lookup.csv IN PLACE: for each row still at
resolution_source == "majority_vote", sets resolved_type / rationale /
confidence and resolution_source = "llm". Rows you've already hand-corrected
(or that apply_manual_llm_resolution.py already touched) are left alone
unless OVERWRITE_EXISTING = True below.

Uses the OpenAI API (gpt-4o family), matching the rest of this project's
extraction pipeline. Requires:
  pip install openai --break-system-packages
  an API key, either pasted into API_KEY below or set as the
  OPENAI_API_KEY environment variable.

Designed for Jupyter/Colab execution. No __main__ guard -- just set the
CONFIG values below and run the whole cell/file.
"""

import json
import os
import time

import pandas as pd

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
LOOKUP_PATH = "entity_type_lookup.csv"
CONTEXT_PATH = "entity_type_ambiguities_context.json"
API_KEY = None    # or paste your key here, e.g. "sk-..."
MODEL = "gpt-4o"
OVERWRITE_EXISTING = False   # True = re-resolve every row, even hand-corrected ones

TAXONOMY_DEFINITIONS = {
    "functional_property": (
        "A macroscopic, application-relevant functional performance outcome "
        "measured in a food-functionality assay: e.g. solubility, water "
        "holding capacity, oil holding capacity, emulsifying activity/"
        "capacity/stability, foaming capacity/stability, gelation ability."
    ),
    "physicochemical_property": (
        "A molecular/physicochemical characteristic describing the protein's "
        "physical or chemical state, not a functional performance outcome: "
        "e.g. surface hydrophobicity, zeta potential, particle size, free "
        "sulfhydryl/amino group content, denaturation temperature."
    ),
    "structural_property": (
        "A structural feature at the secondary/tertiary/quaternary level: "
        "e.g. beta-sheet/alpha-helix/random-coil content, molecular weight, "
        "subunit composition, aggregate or network microstructure."
    ),
    "modification_method": (
        "An active treatment or process applied to an ALREADY-OBTAINED "
        "protein/material to change its properties: e.g. heat treatment, "
        "ultrasonication, pH shifting, enzymatic hydrolysis, glycation, "
        "acetylation, oxidation, high-pressure treatment, drying as a "
        "post-isolation processing step."
    ),
    "extraction_method": (
        "A method used to OBTAIN or ISOLATE the protein from raw biomass in "
        "the first place: e.g. alkaline extraction, isoelectric "
        "precipitation, dry/wet fractionation, solvent extraction, "
        "ultrasound-assisted extraction, hot pressing."
    ),
    "material_used_directly": (
        "A physical substance, ingredient, sample, or reagent referenced as "
        "a noun (not a process): e.g. a named protein isolate/concentrate, "
        "a chemical reagent, an additive. A process name used as a "
        "treatment label (e.g. 'vacuum drying') should still be "
        "modification_method or extraction_method, not this."
    ),
    "rheological_property": (
        "A property characterizing flow or deformation behavior: viscosity, "
        "viscoelasticity, storage/loss modulus (G'/G''), gel "
        "strength/hardness, other textural-mechanical measurements."
    ),
    "sensory_property": (
        "A human-perceived sensory attribute: taste, mouthfeel, sensory "
        "texture perception, appearance/color as judged by a panel."
    ),
}

RESPONSE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "entity_type_resolution",
        "schema": {
            "type": "object",
            "properties": {
                "chosen_type": {"type": "string", "enum": list(TAXONOMY_DEFINITIONS)},
                "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
                "rationale": {"type": "string", "description": "One sentence."},
            },
            "required": ["chosen_type", "confidence", "rationale"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}


def build_prompt(entity_name, ctx):
    breakdown = ctx["type_breakdown"]
    lines = [
        "You are classifying an entity from a plant-protein food-science knowledge graph "
        "into exactly one of a closed 8-category taxonomy. The entity was extracted with "
        "inconsistent type labels across different source sentences; pick the single "
        "best-fitting category based on what the entity actually IS, using the sentences "
        "as evidence.",
        "",
        f'Entity name: "{entity_name}"',
        f"Candidate types seen in the data (with occurrence counts): {breakdown}",
        "",
        "Full taxonomy (choose exactly one):",
    ]
    for t, d in TAXONOMY_DEFINITIONS.items():
        lines.append(f"- {t}: {d}")
    lines.append("")
    lines.append("Evidence sentences, grouped by the type they were assigned under:")
    for t in breakdown:
        examples = ctx["examples_by_type"].get(t, [])
        lines.append(f"\n[{t}]")
        for ex in examples:
            lines.append(f"  - \"{ex['sentence']}\" (doi: {ex['doi']})")
    return "\n".join(lines)


def call_llm(client, prompt):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format=RESPONSE_SCHEMA,
        temperature=0,
    )
    return json.loads(resp.choices[0].message.content)


# ---------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------
lookup_df = pd.read_csv(LOOKUP_PATH)
for col in ["resolved_type", "resolution_source", "confidence", "rationale"]:
    if col in lookup_df.columns:
        lookup_df[col] = lookup_df[col].astype(object)

context = json.loads(open(CONTEXT_PATH).read())
print(f"Loaded {len(lookup_df)} rows from {LOOKUP_PATH}")

key = API_KEY or os.environ.get("OPENAI_API_KEY")
if not key:
    raise SystemExit(
        "No API key. Set API_KEY above or the OPENAI_API_KEY environment variable. "
        "(pip install openai --break-system-packages if needed.)"
    )

from openai import OpenAI
client = OpenAI(api_key=key)

# ---------------------------------------------------------------
# RESOLVE
# ---------------------------------------------------------------
to_resolve = lookup_df if OVERWRITE_EXISTING else lookup_df[lookup_df["resolution_source"] == "majority_vote"]
print(f"Resolving {len(to_resolve)} / {len(lookup_df)} rows with {MODEL} "
      f"({'overwriting all' if OVERWRITE_EXISTING else 'skipping already-corrected rows'})...")

for idx, row in to_resolve.iterrows():
    name = row["entity_name"]
    ctx = context.get(name)
    if ctx is None:
        print(f"  skip {name!r}: not found in context file")
        continue

    prompt = build_prompt(name, ctx)
    decision = None
    for attempt in range(3):
        try:
            decision = call_llm(client, prompt)
            break
        except Exception as e:
            if attempt == 2:
                print(f"  FAILED on {name!r}: {e} -- leaving as majority vote")
            else:
                time.sleep(2 ** attempt)
    if decision is None:
        continue

    lookup_df.loc[idx, "resolved_type"] = decision["chosen_type"]
    lookup_df.loc[idx, "resolution_source"] = "llm"
    lookup_df.loc[idx, "confidence"] = decision["confidence"]
    lookup_df.loc[idx, "rationale"] = decision["rationale"]

    agree = "=" if decision["chosen_type"] == row["majority_type"] else "!="
    print(f"  {name!r}: majority={row['majority_type']} {agree} llm={decision['chosen_type']} ({decision['confidence']})")

# ---------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------
lookup_df.to_csv(LOOKUP_PATH, index=False)
n_llm = (lookup_df["resolution_source"] == "llm").sum()
print(f"\nSaved {LOOKUP_PATH} -- {n_llm} rows now resolved by LLM.")

Loaded 145 rows from entity_type_lookup.csv
Resolving 145 / 145 rows with gpt-4o (skipping already-corrected rows)...
  'protein glutaminase treatment': majority=modification_method = llm=modification_method (high)
  'Chymotrypsin enzyme treatment': majority=modification_method = llm=modification_method (high)
  'NaCl extraction': majority=extraction_method = llm=extraction_method (high)
  '5-day-germinated hemp seed protein isolate with ultrafiltration and structural content analysis': majority=material_used_directly = llm=material_used_directly (high)
  'nanoscale aggregates': majority=material_used_directly != llm=structural_property (high)
  'exposure of hydrophilic regions or residues': majority=structural_property = llm=structural_property (medium)
  'viscoelasticity': majority=functional_property != llm=rheological_property (high)
  'exposure of hydrophobic groups': majority=physicochemical_property = llm=physicochemical_property (high)
  'dense gel network structure': majority=